#### Nombre: Jordan Casamén
#### Asignatura: Implementación de Modelos de Programación Entera

# Minimización de cajas de atención con ventanas de tiempo y períodos


## Motivación

Cuando voy al supermercado, hago las compras, y cuando por fin llego a las cajas toca hacer fila, mientras veo cajas cerradas que perfectamente podrían estar abiertas para que la fila avance más rápido. A veces ni siquiera tengo tanto tiempo, y si la fila está muy larga prefiero dejar la compra para otro momento antes que esperar.

Entonces me pregunté ¿cuántas cajas de cada tipo debería tener abiertas el supermercado en cada momento del día, y a cuál caja y en qué instante debería empezar a atender a cada cliente, para que todos alcancen a ser atendidos dentro del tiempo que están dispuestos a esperar, sin necesidad de abrir más cajas de las que realmente hacen falta?

Este problema no es tan simple como parece ni tiene una fórmula directa que lo resuelva: cada cliente tiene su propio margen de tiempo, hay distintos tipos de caja que no todos los clientes pueden usar, y todo esto cambia a lo largo del día. Por eso se necesita programación lineal entera: permite modelar estas decisiones (qué caja, cuándo empieza cada cliente, cuántas cajas mantener abiertas).

## Descripción del problema

Se considera un supermercado que opera durante una jornada dividida en varios períodos consecutivos. El establecimiento dispone de cajas de distintos tipos, según el tipo de trámite que cada una puede procesar (por ejemplo, cajas rápidas, cajas normales y cajas de autoservicio). Las cajas de un mismo tipo se consideran indistinguibles entre sí; por tanto, únicamente interesa determinar cuántas cajas de cada tipo deben mantenerse operativas en cada período.

Se conoce de antemano un conjunto de clientes que deben ser atendidos durante la jornada. Cada cliente tiene un instante de llegada, una duración de atención requerida y un instante límite antes del cual su atención debe haber finalizado. Además, cada cliente solo puede ser atendido por los tipos de caja compatibles con el trámite que desea realizar.

Mantener operativa una caja de un determinado tipo durante un período genera un costo fijo, independiente del número de clientes atendidos. En cualquier instante, el número de clientes atendidos simultáneamente por un mismo tipo de caja no puede superar el número de cajas operativas de ese tipo. Cada cliente debe iniciar su atención exactamente una vez, en un tipo de caja compatible y en un instante que respete su ventana de tiempo. Si la atención de un cliente se extiende a través de varios períodos consecutivos, la capacidad disponible de ese tipo de caja debe ser suficiente para atenderlo durante toda la duración del servicio.

El problema consiste en determinar, para cada tipo de caja y cada período, el número de cajas que deben mantenerse operativas, así como el tipo de caja y el instante de inicio de atención asignados a cada cliente, de manera que todos los clientes sean atendidos dentro de sus respectivas ventanas de tiempo, respetando las restricciones de capacidad, con el objetivo de minimizar el número total de cajas operativas a lo largo de la jornada.

## Formulación

### Conjuntos

- $I = \{1,\dots,n\}$ es el conjunto de clientes.
- $R = \{\text{ca},\text{cr},\text{cn}\}$ es el conjunto de tipos de caja (autoservicio, rápida, normal).
- $K = \{1,\dots,K\}$ es el conjunto de períodos que particionan la jornada.
- $T$ es el conjunto de instantes discretos de la jornada.
- $T_k \subseteq T$ es el conjunto de instantes que pertenecen al período $k$; la familia $\{T_k\}_{k\in K}$ es una partición de $T$.
- $S_i = \{t\in T: a_i\le t,\ t+p_i\le b_i\}$ es el conjunto de instantes de inicio factibles del cliente $i$.

### Parámetros

- $a_i$ es el instante de llegada del cliente $i$.
- $b_i$ es el instante límite del cliente $i$.
- $p_i$ es la duración de atención requerida por el cliente $i$.
- $c_{ir} \in \{0,1\}$ indica si el cliente $i$ es compatible con el tipo de caja $r$.
- $M_r$ es el número máximo de cajas de tipo $r$ disponibles.

### Variables de decisión

$$x_{irt}=\begin{cases}1, & \text{si el cliente } i \text{ inicia su atención en una caja de tipo } r \text{ en el instante } t,\\ 0, & \text{en caso contrario.}\end{cases} \quad i\in I,\ r\in R,\ t\in S_i$$

$$m_{rk}: \text{número de cajas de tipo } r \text{ que permanecen operativas durante todo el período } k.$$
$$ m_{rk}\in\mathbb{Z}_{\ge0},\ r\in R,\ k\in K$$

### Función objetivo

Minimiza el número total de cajas operativas a lo largo de la jornada, sumado sobre todos los tipos y períodos:

$$\min \sum_{r\in R}\sum_{k\in K} m_{rk}$$

### Restricciones

Cada cliente inicia su atención exactamente una vez, en un tipo de caja compatible y en un instante factible de su ventana:

$$\sum_{r\in R}\sum_{t\in S_i} c_{ir}\,x_{irt} = 1, \qquad \forall i\in I$$

En cualquier instante de un período, el número de clientes atendidos simultáneamente por cajas de tipo $r$ no puede superar el número de cajas de ese tipo que permanecen operativas durante dicho período:

$$\sum_{i\in I}\sum_{\substack{t'\in S_i\\ t'\le t<t'+p_i}} c_{ir}\,x_{irt'} \le m_{rk}, \qquad \forall r\in R,\ \forall k\in K,\ \forall t\in T_k$$

El número de cajas de tipo $r$ operativas en un período no puede superar la cantidad disponible de ese tipo:

$$m_{rk} \le M_r, \qquad \forall r\in R,\ \forall k\in K$$

## Implementación del modelo

### Datos

Los parámetros $p_i$ (duración de atención) se calibraron a partir de rangos de tiempo de pago reportados públicamente por cadenas de supermercados que operan en Ecuador: Mi Comisariato y Supermercados MAS y un artículo de El Universo sobre la adopción de cajas de autopago en Supermaxi, Tía y Megamaxi. Estos rangos (caja de autoservicio: 1 a 2 minutos; caja rápida: 2 a 3 minutos; caja normal: 3 a 4 minutos) se convirtieron en una distribución Gamma por tipo de caja mediante un método de ajuste por percentiles: se fija la media en el punto medio del rango reportado, y se resuelve el parámetro de forma de la Gamma de modo que aproximadamente el 80% de la masa de probabilidad quede contenida dentro de ese rango.

Los parámetros de la dinámica de llegadas (tiempo entre llegadas y perfil horario a lo largo del día) y el tamaño de canasta de cada cliente (usado para determinar su tipo de trámite y, por tanto, su compatibilidad $c_{ir}$) se calibraron a partir de un conjunto de datos reales de transacciones de punto de venta (POS) de un supermercado polaco (Antczak, T., & Weron, R., 2019), disponible públicamente, dado que no se encontró un conjunto de datos equivalente medido en supermercados ecuatorianos. El instante límite $b_i$ no se calibra a partir de datos: se define como una política de servicio (tolerancia máxima de espera desde la llegada), y constituye un parámetro del experimento, no un dato medido.

In [1]:
import pandas as pd
import numpy as np

GRANULARIDAD = 15  # segundos por unidad de tiempo del modelo
TOLERANCIA_ESPERA = 600  # segundos, politica de servicio (tope de espera usado al generar b_i)

# a_i, p_i, b_i: llegada, duracion e instante limite de cada cliente, convertidos a la granularidad del modelo
clientes = pd.read_csv("clientes_simulados.csv").head(500).copy()
clientes["a_i"] = (clientes["a_i"] / GRANULARIDAD).round().astype(int)
clientes["p_i"] = np.maximum(1, np.round(clientes["p_i"] / GRANULARIDAD)).astype(int)
TOL_U = round(TOLERANCIA_ESPERA / GRANULARIDAD)
clientes["b_i"] = clientes["a_i"] + TOL_U + clientes["p_i"]

I = list(clientes["cliente"])
a = dict(zip(clientes["cliente"], clientes["a_i"]))
p = dict(zip(clientes["cliente"], clientes["p_i"]))
b = dict(zip(clientes["cliente"], clientes["b_i"]))
tramite = dict(zip(clientes["cliente"], clientes["tramite"]))

In [2]:
# tipos de caja: autoservicio, rapida, normal
R = ["ca", "cr", "cn"]  
TIPO_DE_TRAMITE = {"rapida": "cr", "normal": "cn"}

# M[r]: cajas de tipo r fisicamente disponibles (dato real: 4 autoservicio, 2 rapidas, 6 normales)
M = {"ca": 4, "cr": 2, "cn": 6}

# c[i,r]: compatibilidad cliente-tipo de caja
c = {(i, r): int(r == "ca" or r == TIPO_DE_TRAMITE[tramite[i]]) for i in I for r in R}  

# S[i]: instantes de inicio factibles del cliente i
S = {i: range(a[i], b[i] - p[i] + 1) for i in I}  

 # duracion de un periodo
L = round(15 * 60 / GRANULARIDAD) 
H = max(b.values())
# T: instantes discretos de toda la jornada
T = range(H + 1)  
# K: periodos consecutivos
K = range(H // L + 1)  
# T_k: instantes que pertenecen al periodo k
T_k = {k: range(k * L, min((k + 1) * L, H + 1)) for k in K}  

# dominio de la variable x_irt
idx = [(i, r, t) for i in I for r in R if c[i, r] == 1 for t in S[i]]  

# activos[r, t]: pares (i, t') de clientes compatibles con el tipo r cuyo intervalo de atencion [t', t'+p_i) cubre el instante t
activos = {(r, t): [(i, tp) for i in I for tp in S[i] if tp <= t < tp + p[i] and c[i, r] == 1] for r in R for t in T}

### Modelo

In [3]:
import gurobipy as gp
from gurobipy import GRB

modelo = gp.Model("Cajas_Supermercado")
x = modelo.addVars(idx, vtype=GRB.BINARY, name="x")
m = modelo.addVars(R, K, vtype=GRB.INTEGER, lb=0, name="m")

modelo.setObjective(m.sum(), GRB.MINIMIZE)

modelo.addConstrs((x.sum(i, "*", "*") == 1 for i in I), "asignacion")
modelo.addConstrs((gp.quicksum(x[i, r, tp] for (i, tp) in activos[r, t]) <= m[r, k] for r in R for k in K for t in T_k[k]), "capacidad")
modelo.addConstrs((m[r, k] <= M[r] for r in R for k in K), "capacidad_maxima")

modelo.Params.TimeLimit = 180
modelo.optimize()

Set parameter Username
Set parameter LicenseID to value 2815204
Academic license - for non-commercial use only - expires 2027-04-29
Set parameter TimeLimit to value 180
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 3750H with Radeon Vega Mobile Gfx, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  180

Optimize a model with 3785 rows, 41054 columns and 531037 nonzeros (Min)
Model fingerprint: 0x6925a5ae
Model has 54 linear objective coefficients
Variable types: 0 continuous, 41054 integer (41000 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 6e+00]

Presolve removed 892 rows and 1128 columns
Presolve time: 1.39s
Presolved: 2893 rows, 39926 columns, 514318 nonzeros
Variable types: 0 continuous, 39926 integer (39884 binary

In [4]:
print(f"Cajas necesarias en total: {modelo.objVal:.0f}")
print("Desglose por tipo y periodo:")
print("\n".join(f"  periodo {k}: " + ", ".join(f"{r}={round(m[r, k].x):.0f}" for r in R) for k in K))

asignacion = {i: (r, t) for (i, r, t) in idx if x[i, r, t].x >= 0.5}
print(f"\nClientes atendidos en total: {len(asignacion)}")
print("Desglose por tipo:")
print("\n".join(f"  {r}: {sum(1 for r2, t in asignacion.values() if r2 == r)}" for r in R))

espera = {i: (asignacion[i][1] - a[i]) * GRANULARIDAD for i in I}
print(f"\nTiempo de espera promedio en total: {sum(espera.values()) / len(espera):.1f} s")
print("Desglose por tipo:")
print("\n".join( f"  {r}: {sum(espera[i] for i in I if asignacion[i][0] == r) / sum(1 for i in I if asignacion[i][0] == r):.1f} s" for r in R ))
servicio = {i: p[i] * GRANULARIDAD for i in I}
print(f"\nTiempo en caja promedio en total: {sum(servicio.values()) / len(servicio):.1f} s")
print("Desglose por tipo:")
print("\n".join( f"  {r}: {sum(servicio[i] for i in I if asignacion[i][0] == r) / sum(1 for i in I if asignacion[i][0] == r):.1f} s" for r in R))

total_sistema = {i: espera[i] + servicio[i] for i in I}
print(f"\nTiempo total en el sistema (fila + caja) promedio en total: {sum(total_sistema.values()) / len(total_sistema):.1f} s")
print("Desglose por tipo:")
print("\n".join(
    f"  {r}: {sum(total_sistema[i] for i in I if asignacion[i][0] == r) / sum(1 for i in I if asignacion[i][0] == r):.1f} s"
    for r in R
))

Cajas necesarias en total: 104
Desglose por tipo y periodo:
  periodo 0: ca=1, cr=0, cn=0
  periodo 1: ca=1, cr=0, cn=0
  periodo 2: ca=1, cr=0, cn=0
  periodo 3: ca=0, cr=0, cn=0
  periodo 4: ca=2, cr=0, cn=1
  periodo 5: ca=4, cr=2, cn=2
  periodo 6: ca=2, cr=0, cn=2
  periodo 7: ca=3, cr=0, cn=2
  periodo 8: ca=2, cr=0, cn=1
  periodo 9: ca=3, cr=1, cn=1
  periodo 10: ca=3, cr=1, cn=1
  periodo 11: ca=4, cr=1, cn=3
  periodo 12: ca=4, cr=2, cn=6
  periodo 13: ca=4, cr=2, cn=6
  periodo 14: ca=4, cr=2, cn=5
  periodo 15: ca=3, cr=2, cn=4
  periodo 16: ca=4, cr=2, cn=6
  periodo 17: ca=1, cr=1, cn=2

Clientes atendidos en total: 500
Desglose por tipo:
  ca: 232
  cr: 94
  cn: 174

Tiempo de espera promedio en total: 298.2 s
Desglose por tipo:
  ca: 307.4 s
  cr: 300.6 s
  cn: 284.7 s

Tiempo en caja promedio en total: 178.1 s
Desglose por tipo:
  ca: 166.2 s
  cr: 149.8 s
  cn: 209.2 s

Tiempo total en el sistema (fila + caja) promedio en total: 476.3 s
Desglose por tipo:
  ca: 473.5 

### Interpretación de los resultados

El mínimo de cajas a usar durante toda la jornada es de 104 caja-períodos, para atender a los 500 clientes de la instancia. De esos 104: autoservicio aporta 46 (abre en 17 de los 18 períodos, todos menos el período 3), rápida aporta 16 (abre en 10 de los 18 períodos: el 5 y del 9 al 17), y normal aporta 42 (abre en 14 de los 18 períodos: del 4 al 17).

En cuanto a clientes atendidos: autoservicio atiende a 232, rápida a 94 y normal a 174. Autoservicio y normal atienden cantidades de clientes bastante más parecidas entre sí que en instancias más pequeñas, lo cual es consistente con que ambos permanecen abiertos durante la mayor parte de la jornada (17 y 14 de los 18 períodos, respectivamente), mientras que rápida, al abrir en menos períodos (10 de 18) y con menor capacidad máxima (2 cajas), atiende proporcionalmente a menos gente.

Sobre cuántas cajas se abren realmente en los momentos de mayor carga: a diferencia de instancias con menos clientes, aquí los tres tipos llegan a su tope máximo de capacidad en algún momento del día. Autoservicio llega a sus 4 cajas simultáneas en 6 de los 18 períodos, rápida llega a sus 2 cajas en 6 períodos, y normal llega a sus 6 cajas en 3 períodos (los períodos 12, 13 y 16, los de mayor carga del día). Que los tres tipos toquen su tope en algún momento indica que, con 500 clientes, la capacidad instalada (12 cajas en total) está siendo utilizada casi al máximo, sin margen sobrante en ningún tipo.

Sobre los tiempos (en promedio, sobre los 500 clientes): un cliente espera en promedio 298.2 segundos (cerca de 5 minutos) en la fila antes de que le toque su turno, y la atención en sí dura en promedio 178.1 segundos (cerca de 3 minutos), para un tiempo total en el sistema de 476.3 segundos (cerca de 8 minutos) desde que el cliente llega hasta que termina de pagar. Por tipo, la espera es bastante similar entre los tres (entre 284.7 y 307.4 segundos), pero la duración de la atención sí varía de forma notoria: normal es el más lento (209.2 s, coherente con que procesa los trámites más largos) y rápida el más rápido (149.8 s).

## Análisis de sensibilidad

In [5]:
def construir(n, multiplicador_p=1.0):
    clientes = pd.read_csv("clientes_simulados.csv").head(n).copy()
    clientes["a_i"] = (clientes["a_i"] / GRANULARIDAD).round().astype(int)
    clientes["p_i"] = np.maximum(1, np.round(clientes["p_i"] / GRANULARIDAD * multiplicador_p)).astype(int)
    clientes["b_i"] = clientes["a_i"] + TOL_U + clientes["p_i"]

    I = list(clientes["cliente"])
    a = dict(zip(clientes["cliente"], clientes["a_i"]))
    p = dict(zip(clientes["cliente"], clientes["p_i"]))
    b = dict(zip(clientes["cliente"], clientes["b_i"]))
    tramite = dict(zip(clientes["cliente"], clientes["tramite"]))

    c = {(i, r): int(r == "ca" or r == TIPO_DE_TRAMITE[tramite[i]]) for i in I for r in R}
    S = {i: range(a[i], b[i] - p[i] + 1) for i in I}

    H = max(b.values())
    T = range(H + 1)
    K = range(H // L + 1)
    T_k = {k: range(k * L, min((k + 1) * L, H + 1)) for k in K}

    idx = [(i, r, t) for i in I for r in R if c[i, r] == 1 for t in S[i]]
    activos = {(r, t): [(i, tp) for i in I for tp in S[i] if tp <= t < tp + p[i] and c[i, r] == 1] for r in R for t in T}

    return I, c, K, T_k, idx, activos


def resolver(I, c, K, T_k, idx, activos, M_tipo, time_limit):
    modelo = gp.Model("Cajas_Supermercado")
    modelo.Params.OutputFlag = 0
    x = modelo.addVars(idx, vtype=GRB.BINARY, name="x")
    m = modelo.addVars(R, K, vtype=GRB.INTEGER, lb=0, name="m")

    modelo.setObjective(m.sum(), GRB.MINIMIZE)
    modelo.addConstrs((x.sum(i, "*", "*") == 1 for i in I), "asignacion")
    modelo.addConstrs((gp.quicksum(x[i, r, tp] for (i, tp) in activos[r, t]) <= m[r, k] for r in R for k in K for t in T_k[k]), "capacidad")
    modelo.addConstrs((m[r, k] <= M_tipo[r] for r in R for k in K), "capacidad_maxima")

    modelo.Params.TimeLimit = time_limit
    modelo.optimize()
    return modelo


def resumen(modelo):
    if modelo.SolCount > 0:
        return f"{modelo.objVal:.0f} caja-periodos, gap {modelo.MIPGap:.2%}"
    if modelo.Status == GRB.INFEASIBLE:
        return "infactible (demostrado)"
    return "sin solucion encontrada en el tiempo limite (no se demostro infactibilidad)"

### Sensibilidad a la demanda

In [ ]:
print("Cantidad de clientes -> resultado")
print("\n".join(
    f"  n={n}: {resumen(resolver(*construir(n), M, 180))}"
    for n in [200, 400, 500, 800, 1000, 1500]
))

Cantidad de clientes -> resultado


### Comparación: aumentando cajas normales (ca:4, cr:2, cn:10; total 16 cajas en vez de 12)

In [ ]:
M_comparacion = {"ca": 4, "cr": 2, "cn": 10}

print("Cantidad de clientes -> resultado (ca:4, cr:2, cn:10; total 16 cajas en vez de 12)")
print("\n".join( f"  n={n}: {resumen(resolver(*construir(n), M_comparacion, 180))}" for n in [200, 400, 500, 800, 1000, 1500]))

Cantidad de clientes -> resultado (ca:4, cr:2, cn:10; total 16 cajas en vez de 12)
  n=200: 42 caja-periodos, gap 2.38%
  n=400: 82 caja-periodos, gap 0.00%
  n=500: 105 caja-periodos, gap 3.81%
  n=800: 170 caja-periodos, gap 4.71%
  n=1000: 281 caja-periodos, gap 28.47%
  n=1500: 515 caja-periodos, gap 41.75%


### Sensibilidad al tiempo de servicio

In [ ]:
print("Multiplicador de p_i -> resultado (n=500 fijo)")
print("\n".join( f"  x{mult}: {resumen(resolver(*construir(500, mult), M, 180))}" for mult in [1.0, 1.2, 1.5, 2.0]))

Multiplicador de p_i -> resultado (n=500 fijo)
  x1.0: 104 caja-periodos, gap 2.88%
  x1.2: sin solucion encontrada en el tiempo limite (no se demostro infactibilidad)
  x1.5: infactible (demostrado)
  x2.0: infactible (demostrado)


### Interpretación de la sensibilidad

La sensibilidad a la demanda muestra que, con la capacidad real (4 autoservicio, 2 rápida, 6 normal = 12 cajas en total), el modelo sigue encontrando una asignación válida incluso con demandas bastante mayores a la instancia base: a 800 clientes necesita 168 caja-períodos, y a 1000 clientes, 258 caja-períodos, aunque en este último caso el gap de optimalidad sube a 22.09%, señal de que el problema se vuelve mucho más difícil de resolver a esa escala. A 1500 clientes el solver no logra encontrar ninguna asignación dentro del tiempo asignado (180 segundos), pero esto no equivale a que el problema sea infactible: el propio caso de 1000 clientes se comportaría igual con menos tiempo de cómputo, y solo se resolvió al darle más tiempo. Es decir, con los recursos de cómputo usados aquí no se puede afirmar que 1500 clientes superen la capacidad real del supermercado, solo que no se logró determinar una asignación factible en ese caso dentro del tiempo disponible.

En esta comparación se aumenta la capacidad total instalada: se pasa de 12 cajas en total (4 autoservicio + 2 rápida + 6 normal) a 16 (4 autoservicio + 2 rápida + 10 normal), manteniendo fijas autoservicio y rápida y subiendo solo normal de 6 a 10 — no se trata de redistribuir cajas entre tipos, sino de agregar más cajas normales sobre la misma base. Con esta capacidad ampliada, la infactibilidad desaparece por completo: incluso 1500 clientes obtienen una asignación válida (515 caja-períodos, con un gap de 41.75% dada la dificultad computacional de esa instancia). Esto indica que, de los tres tipos, normal es el que actúa como cuello de botella real del sistema para demandas altas: reforzar autoservicio o rápida no habría bastado, porque el límite que impedía atender más clientes estaba específicamente en la capacidad de cajas normales.

Sobre el tiempo de servicio: partiendo de los mismos 500 clientes que ya usan casi toda la capacidad instalada (como se vio en la interpretación de los resultados, los tres tipos de caja llegan a su tope en algún momento del día), aumentar apenas un 20% la duración de atención ya hace que el solver no logre encontrar ninguna asignación dentro del tiempo asignado. Con aumentos mayores (x1.5 y x2.0) el modelo sí logra demostrar formalmente que no existe ninguna asignación posible: el problema es infactible, no solo difícil de resolver. Esto tiene sentido porque, a diferencia de la sensibilidad a la demanda (donde agregar clientes puede seguir usando los mismos horarios ya ocupados por otros), aquí cada cliente ya existente ocupa una caja durante más tiempo, y como el sistema no tenía ningún margen sobrante en ningún tipo de caja, no hay forma de compensar esa demora sin violar las ventanas de tiempo de los clientes.


## Referencias

- Kravchenko, S. A., & Werner, F. (2009). Minimizing the number of machines for scheduling jobs with equal processing times. *European Journal of Operational Research*, 199(3), 595–600. https://doi.org/10.1016/j.ejor.2008.10.008
- Antczak, T., & Weron, R. (2019). Point of Sale (POS) Data from a Supermarket: Transactions and Cashier Operations. *Data*, 4(2), 67. https://doi.org/10.3390/data4020067
- El Universo. Usted mismo toma su pedido o escanea sus productos y los paga: cajas de autoservicio reducen tiempo en fila y van ganando espacios en cadenas de comida rápida y supermercados. https://www.eluniverso.com/noticias/economia/usted-mismo-toma-su-pedido-o-escanea-sus-productos-y-los-paga-cajas-autoservicio-reducen-tiempo-en-fila-y-van-ganando-espacios-en-cadenas-de-comida-rapida-y-supermercados-nota/
